# Model comparison: log-likelihood, recall, precision as in the HuProSO3 paper

The unconditional pose prior of the HuProSO3 paper (Dünkel, Salzmann, Pfaff) is evaluated with recall, precision (Tab. 1, Tab. 6) and the log-likelihood (Tab. 7). This notebook prints the paper's tables and evaluates our models with the **same protocol, the same sample sizes and the same units** (`hp/evaluation/huproso3_protocol.py`):

| metric | protocol of the paper | unit |
|---|---|---|
| **recall** (Tab. 1) | 1k dataset poses vs. **100k** model samples; distance of each dataset pose to its nearest sample | **sum** over the 19 joints of the geodesic distance (Eq. 9), **radians**; mean [median] |
| **precision** (Tab. 1) | 1k model samples vs. **100k** dataset poses; distance of each sample to its nearest dataset pose | the same |
| recall / precision (Tab. 6) | the same poses, distance = MPJPE after forward kinematics | mm |
| **log-likelihood** (Tab. 7) | mean over 5000 dataset poses (`num_eval_samples` of the paper's config) | nats, relative to the uniform distribution on SO(3)^19 |

Every column is computed on the test split ("Test") and on the train split ("Train"). The dataset poses are drawn with a fixed seed, so **every model is evaluated on the same dataset poses**. Additions the paper does not have: 3 repeats with independent draws (± std over the repeats), and a yardstick row: 100k real train poses (other subjects) in place of a model.

What cannot be made the same:
- **The data.** Ours is the SMPL-X **G** release of AMASS; the authors most likely used the **N** release. Even the authors' prior `id_0900` does not reproduce the paper's numbers here: its row shows how much the data alone moves them. Compare the models with each other on our data, not with the paper's absolute numbers.
- **Train split:** every 30th frame of our train split (`AMASS_rotations_c30`, 497k poses; the 100k pool is drawn from it). For the authors' prior our train split is unseen data, so its "Train" numbers are no train numbers.
- **ODE models** (RSGM, PoseRFM): the log-likelihood comes from the probability flow ODE with `LIKELIHOOD_STEPS` midpoint steps (200: ~0.06 nats too high, measured on RSGM c30).

Reports are saved per model in `REPORT_DIR` (`cache/<notebook name>` in the repository, git-ignored) and reused while the settings are the same; `RECOMPUTE = True` evaluates again. The settings keys are those of `rsgm_performance_tests.ipynb`, so its reports (`~/data/experiments/huproso3_protocol`) can be copied here and are reused.

In [1]:
import json
from pathlib import Path

import numpy as np
import torch

from hp.evaluation.data import load_reference
from hp.evaluation.huproso3_protocol import PAPER, PaperProtocolConfig, evaluate_paper_protocol
from hp.models.io import MANIFEST, load_model
from hp.training import best_checkpoint
from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
# A run of the model zoo (a run folder, a checkpoint folder, or a folder of runs: the newest finished one is taken)
# and what it was trained on. Of a run, CHECKPOINT says which weights are evaluated (a checkpoint folder is taken as it is). The flows of SO3PriorTrainer are the copies converted by rsgm_huproso_comparison.ipynb.
MODELS = {
    "RSGM c30":               dict(run="~/data/experiments/rsgm_c30", weights="weights_ema", data="every 30th frame"),
    "RSGM full t23 AdamW":    dict(run="~/data/experiments/rsgm_tuned_r1t23_adamw", weights="weights_ema", data="all frames"),
    "flow c30":               dict(run="~/data/experiments/so3_flow_c30/id_9030", weights="weights", data="every 30th frame"),
    "flow full":              dict(run="~/data/experiments/prior_full/run_0001", weights="weights", data="all frames"),  # no EMA
    "PoseRFM full":           dict(run="~/data/experiments/poserfm", weights="weights_ema", data="all frames"),
    # the run before it broke down at step 33,866: its 30k checkpoint (also its best one)
    "PoseRFM full t24 @30k":  dict(run="~/data/experiments/poserfm-full-t24/run_0001/checkpoint_030000", weights="weights_ema",
                                   data="all frames"),
    "flow authors (id_0900)": dict(run="~/data/experiments/so3_flow_authors/id_0900", weights="weights", data="authors' AMASS"),
}
YARDSTICK = "real train poses (other subjects)"  # 100k real train poses in place of a model, on the test split
TEST_ROTATIONS = "~/data/AMASS_rotations"        # all frames
# TRAIN_ROTATIONS = "~/data/AMASS_rotations_c30"   # every 30th frame of the train split
TRAIN_ROTATIONS = "~/data/AMASS_rotations"   ## all frames
SPLIT = "original"                               # the official AMASS split
DEVICES = {"rsgm": "mps", "poserfm": "mps", "so3_flow": "cpu"}  # by model type
ODE_MODELS = ("rsgm", "poserfm")                 # log-likelihood by the probability flow ODE
LIKELIHOOD_STEPS = 200                           # its midpoint steps (the configs have 100: ~0.3 nats too high)
PROTOCOL = PaperProtocolConfig(n_queries=1_000, n_pool=100_000, n_likelihood=5_000, repeats=3, seed=0)  # the paper's sizes
REPORT_DIR = REPO_ROOT / "cache" / "model_comparison_loglkhd_recall_precision"  # one report per model; reused while the settings are the same
CHECKPOINT = "best"  # "best": the kept checkpoint with the lowest validation loss of the EMA weights (hp.training.best_checkpoint),
                     # the final weights if the run has none ranked (older runs, flows of SO3PriorTrainer); "final": the final weights
RECOMPUTE = False
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
REPORT_DIR = expand(REPORT_DIR)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def run_folder(path):
    path = expand(path)
    if (path / MANIFEST).is_file():
        return path
    runs = sorted(p for p in path.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / MANIFEST).is_file())
    return runs[-1] if runs else None


def weights_folder(run):
    """The folder whose weights are evaluated, and what it is: the best checkpoint of the run, or the run (its final weights)."""
    if CHECKPOINT == "best" and not run.name.startswith("checkpoint_"):
        best = best_checkpoint(run)
        if best is not None:
            metric = json.loads((best / "metric.json").read_text())
            return best, f"best: {best.name}, {metric['kind']} loss {metric['value']:.4f}"
        return run, "final (no ranked checkpoints)"
    return run, "the checkpoint given" if run.name.startswith("checkpoint_") else "final"

## 1. The tables of the paper

As printed: Tab. 1 (summed geodesic distance, radians), Tab. 6 (MPJPE, mm), Tab. 7 (log-likelihood of the unconditional prior). Mean [median]; lower distances and higher log-likelihoods are better.

In [2]:
COLUMNS = [(s, m) for s in ("test", "train") for m in ("recall", "precision")]
HEAD = f"{'':34s}" + "".join(f"{s.capitalize() + ' ' + m:>24s}" for s, m in COLUMNS)


def paper_table(kind):
    print(HEAD)
    for name, r in PAPER[kind].items():
        print(f"{name:34s}" + "".join(f"{r[s][m][0]:>12.2f} [{r[s][m][1]:5.2f}]    " for s, m in COLUMNS))


print("Tab. 1: summed geodesic distance [rad], mean [median]")
paper_table("geodesic_rad")
print("\nTab. 6: MPJPE [mm], mean [median]")
paper_table("mpjpe_mm")
print("\nTab. 7: log-likelihood of the unconditional prior")
print(f"{'':34s}{'Prior (test)':>16s}{'Prior (Train)':>16s}")
for name, r in PAPER["log_likelihood"].items():
    print(f"{name:34s}{r['test']:16.1f}{r['train']:16.1f}")

Tab. 1: summed geodesic distance [rad], mean [median]
                                               Test recall          Test precision            Train recall         Train precision
GAN-S                                     3.76 [ 3.34]            4.51 [ 4.23]            3.57 [ 3.34]            4.38 [ 4.13]    
6D NF                                     3.66 [ 3.16]            4.50 [ 4.00]            3.55 [ 3.32]            4.42 [ 4.10]    
HuProSO3                                  3.44 [ 2.95]            4.24 [ 3.71]            2.93 [ 2.64]            3.90 [ 3.59]    

Tab. 6: MPJPE [mm], mean [median]
                                               Test recall          Test precision            Train recall         Train precision
GAN-S                                    50.60 [48.40]           68.70 [64.20]           39.70 [35.00]           56.20 [49.70]    
6D NF                                    46.30 [39.90]           65.80 [57.80]           39.40 [34.20]           57.20 [49.40

## 2. The dataset poses

In [3]:
splits = {
    "test": load_reference("test", TEST_ROTATIONS, SPLIT),
    "train": load_reference("train", TRAIN_ROTATIONS, SPLIT),
}
for name, poses in splits.items():
    print(f"{name}: {len(poses):,} poses")

test: 117,679 poses
train: 14,779,964 poses


## 3. Evaluate

100k samples per model, then recall, precision and the log-likelihood on both splits. The weights of every model are printed first (`CHECKPOINT`). From scratch the ODE models take the longest (the log-likelihood of 2 × 3 × 5000 poses); saved reports are reused.

In [4]:
def settings(extra=None):
    return dict(protocol=PROTOCOL.__dict__, test=str(TEST_ROTATIONS), train=str(TRAIN_ROTATIONS), split=SPLIT, **(extra or {}))


def cached(name, settings_, compute):
    path = REPORT_DIR / f"{name.replace(' ', '_').replace('(', '').replace(')', '')}.json"
    if path.is_file() and not RECOMPUTE:
        report = json.loads(path.read_text())
        if report.get("settings") == json.loads(json.dumps(settings_)):
            print(f"--- {name}: saved report {path}")
            return report
    report = compute()
    report["settings"] = settings_
    path.write_text(json.dumps(report))
    print(f"--- {name}: evaluated, saved to {path}")
    return report


reports, chosen = {}, {}
for name, m in MODELS.items():
    run = run_folder(m["run"])
    if run is None:
        print(f"{name}: skipped, no finished run in {m['run']}")
        continue
    run, chosen[name] = weights_folder(run)
    print(f"{name}: {run} ({chosen[name]})")
    kind = json.loads((run / MANIFEST).read_text())["build_args"]["name"]
    steps = LIKELIHOOD_STEPS if kind in ODE_MODELS else None

    def compute(run=run, kind=kind, weights=m["weights"], steps=steps):
        model = load_model(run, device=DEVICES[kind], weights=weights)
        if steps is not None:
            model.likelihood_steps = steps
        return evaluate_paper_protocol(model, splits, PROTOCOL, progress=True)

    # the same keys as rsgm_performance_tests.ipynb, so that its reports can be reused
    reports[name] = cached(name, settings(dict(run=str(run), weights=m["weights"], kind=kind, likelihood_steps=steps)), compute)

rng = np.random.default_rng(1)
pool = splits["train"][rng.choice(len(splits["train"]), PROTOCOL.n_pool, replace=False)]
reports[YARDSTICK] = cached(YARDSTICK, settings(dict(pool_seed=1)),
                            lambda: evaluate_paper_protocol(None, {"test": splits["test"]}, PROTOCOL, generated=pool, progress=True))

RSGM c30: /Users/evgeniifedorov/data/experiments/rsgm_c30/run_0001 (final (no ranked checkpoints))


sampling:   0%|          | 0/20 [00:00<?, ?batch/s]

precision / recall:   0%|          | 0/3 [00:00<?, ?repeat/s]

log-likelihood:   0%|          | 0/15000 [00:00<?, ?pose/s]

precision / recall:   0%|          | 0/3 [00:00<?, ?repeat/s]

...

precision / recall:   0%|          | 0/3 [00:00<?, ?repeat/s]

log-likelihood:   0%|          | 0/15000 [00:00<?, ?pose/s]

--- flow authors (id_0900): evaluated, saved to /Users/evgeniifedorov/Documents/HuProSO/cache/model_comparison_loglkhd_recall_precision/flow_authors_id_0900.json


precision / recall:   0%|          | 0/3 [00:00<?, ?repeat/s]

--- real train poses (other subjects): evaluated, saved to /Users/evgeniifedorov/Documents/HuProSO/cache/model_comparison_loglkhd_recall_precision/real_train_poses_other_subjects.json


## 4. Recall and precision next to Tab. 1 and Tab. 6

The paper's rows first, then ours: mean [median] ± the std of the mean over the 3 repeats. Same sizes (1k vs. 100k) and units as the paper. The yardstick has no train columns (its poses are the train poses).

In [5]:
def cell(report, split, metric):
    if split not in report:
        return f"{'-':>24s}"
    r = report[split][metric]
    return f"{r['mean']['mean']:7.2f} [{r['median']['mean']:6.2f}] ±{r['mean']['std']:4.2f}"


def table(kind, unit):
    print(HEAD)
    for name, r in PAPER[kind].items():
        print(f"{'paper: ' + name:34s}" + "".join(f"{r[s][m][0]:>12.2f} [{r[s][m][1]:5.2f}]    " for s, m in COLUMNS))
    print("-" * len(HEAD))
    for name, r in reports.items():
        print(f"{name:34.33s}" + "".join(f"{cell(r, s, f'{m}_{unit}'):>24s}" for s, m in COLUMNS))


print("summed geodesic distance [rad] (Tab. 1)\n")
table("geodesic_rad", "geodesic")
print("\n\nMPJPE [mm] (Tab. 6)\n")
table("mpjpe_mm", "mpjpe")

summed geodesic distance [rad] (Tab. 1)

                                               Test recall          Test precision            Train recall         Train precision
paper: GAN-S                              3.76 [ 3.34]            4.51 [ 4.23]            3.57 [ 3.34]            4.38 [ 4.13]    
paper: 6D NF                              3.66 [ 3.16]            4.50 [ 4.00]            3.55 [ 3.32]            4.42 [ 4.10]    
paper: HuProSO3                           3.44 [ 2.95]            4.24 [ 3.71]            2.93 [ 2.64]            3.90 [ 3.59]    
----------------------------------------------------------------------------------------------------------------------------------
RSGM c30                               3.70 [  3.30] ±0.03     4.45 [  4.27] ±0.03     3.39 [  3.12] ±0.02     3.44 [  3.23] ±0.02
RSGM full t23 AdamW                    3.60 [  3.22] ±0.03     4.26 [  4.08] ±0.03     3.20 [  2.95] ±0.02     3.18 [  3.01] ±0.04
flow c30                               3.6

## 5. Log-likelihood next to Tab. 7


Mean log-likelihood of 5000 dataset poses (relative to the uniform distribution on SO(3)^19, higher is better) ± the std over the repeats, and the median. A mean far from the median, or a non-zero implausible share (per-pose values outside [0, 250]), means that a few broken values carry the mean.

In [6]:
print(f"{'':28s} {'trained on':18s} {'Test: mean':>16s} {'median':>8s} {'implaus.':>9s} {'Train: mean':>16s} {'median':>8s} {'implaus.':>9s}")
for name, v in PAPER["log_likelihood"].items():
    print(f"{'paper: ' + name:28s} {'authors AMASS':18s} {v['test']:16.1f} {'':8s} {'':9s} {v['train']:16.1f}")
print("-" * 120)
for name, r in reports.items():
    if name == YARDSTICK:
        continue
    row = f"{name:28.27s} {MODELS[name]['data']:18.18s}"
    for split in ("test", "train"):
        ll = r[split]["likelihood"]
        row += f" {ll['mean']['mean']:9.2f} ±{ll['mean']['std']:5.2f} {ll['median']['mean']:8.2f} {ll['implausible_share']['mean']:9.4f}"
    print(row)

                             trained on               Test: mean   median  implaus.      Train: mean   median  implaus.
paper: HuProSO3              authors AMASS                 137.6                               184.7
------------------------------------------------------------------------------------------------------------------------
RSGM c30                     every 30th frame      137.00 ± 0.04   142.16    0.0000    152.97 ± 0.30   154.19    0.0000
RSGM full t23 AdamW          all frames            138.39 ± 0.08   143.16    0.0000    160.53 ± 0.36   160.92    0.0000
flow c30                     every 30th frame      135.23 ± 0.06   138.50    0.0000    167.87 ± 0.53   166.27    0.0056
flow full                    all frames            134.79 ± 0.07   137.92    0.0000    171.96 ± 0.50   170.28    0.0079
PoseRFM full                 all frames            137.62 ± 0.02   142.32    0.0000    162.21 ± 0.41   161.90    0.0000
PoseRFM full t24 @30k        all frames            139.38 

## 6. Summary: the columns of Tab. 1 and Tab. 7 in one table

Means only (geodesic in rad, log-likelihood in nats). Our rows are on our data (SMPL-X G), the paper's on the authors' data.

In [7]:
cols = ["LL test", "LL train", "R test", "P test", "R train", "P train"]
print(f"{'':34s}" + "".join(f"{c:>10s}" for c in cols))
p, g = PAPER["log_likelihood"]["HuProSO3"], PAPER["geodesic_rad"]["HuProSO3"]
print(f"{'paper: HuProSO3':34s}{p['test']:10.1f}{p['train']:10.1f}" + "".join(f"{g[s][m][0]:10.2f}" for s, m in COLUMNS))
print("-" * 94)
fmt = lambda x: f"{x:10.2f}" if x is not None else f"{'-':>10s}"
summary = {}
for name, r in reports.items():
    ll = [r[s]["likelihood"]["mean"]["mean"] if s in r and r[s].get("likelihood") else None for s in ("test", "train")]
    geo = [r[s][f"{m}_geodesic"]["mean"]["mean"] if s in r else None for s, m in COLUMNS]
    summary[name] = dict(zip(cols, ll + geo))
    print(f"{name:34.33s}" + "".join(fmt(x) for x in ll + geo))

(REPORT_DIR / "model_comparison_summary.json").write_text(json.dumps({"paper": PAPER, "models": summary}, indent=2))
print(f"\nsaved to {REPORT_DIR / 'model_comparison_summary.json'}")

                                     LL test  LL train    R test    P test   R train   P train
paper: HuProSO3                        137.6     184.7      3.44      4.24      2.93      3.90
----------------------------------------------------------------------------------------------
RSGM c30                              137.00    152.97      3.70      4.45      3.39      3.44
RSGM full t23 AdamW                   138.39    160.53      3.60      4.26      3.20      3.18
flow c30                              135.23    167.87      3.61      4.49      3.18      3.56
flow full                             134.79    171.96      3.60      4.43      3.10      3.40
PoseRFM full                          137.62    162.21      3.61      4.70      3.10      3.56
PoseRFM full t24 @30k                 139.38    173.05      3.66      4.70      2.97      3.46
flow authors (id_0900)                133.04    146.60      3.72      4.89      3.41      3.89
real train poses (other subjects)          -      